# Day 20: Refactoring RAG with Qdrant as a Permanent Storage Layer

## Core Theory (Just-in-Time)

Up until now, our Retrieval-Augmented Generation (RAG) system relied on in-memory storage (like basic dictionaries or lists) for storing vector embeddings. While this works for prototyping and small datasets, it fails in production. In-memory data is lost when the application restarts, and linear search through vectors scales horribly $O(N)$ as your dataset grows.

To build a production-grade AI application, we need a **Vector Database**. 

**Why Qdrant?**
Qdrant is a high-performance, open-source vector similarity search engine written in Rust. It offers:
- **Persistence:** Embeddings are saved to disk (or a dedicated server/cloud cluster) and survive application restarts.
- **Fast Search:** It uses HNSW (Hierarchical Navigable Small World) algorithms to achieve sub-linear $O(log N)$ approximate nearest neighbor (ANN) search.
- **Filtering:** You can attach rich JSON payloads to your vectors and filter search results by metadata before computing vector distances.

**How it fits into RAG:**
1. **Ingestion:** Text is chunked, embedded using an Embedding Model, and stored in Qdrant along with its original text as payload.
2. **Retrieval:** A user query is embedded into a vector. We query Qdrant for the `k` closest vectors (cosine similarity).
3. **Generation:** The original text from the Qdrant payload is extracted and passed to the LLM to generate the final response.


In [1]:
from typing import List, Dict, Any
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
from langchain_core.embeddings import Embeddings
from pydantic import BaseModel, Field
import uuid

class Document(BaseModel):
    """Represents a chunk of text to be stored and retrieved."""
    id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    content: str
    metadata: Dict[str, Any] = Field(default_factory=dict)

class MockEmbeddings(Embeddings):
    """
    A deterministic mock embedding model for demonstration purposes.
    In production, replace this with OpenAIEmbeddings, HuggingFaceEmbeddings, etc.
    """
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # Returns a vector where the length is 3 (dimensionality)
        # Simply encoding string length as a deterministic feature for testing.
        return [[len(text) * 0.1, 0.5, 0.5] for text in texts]

    def embed_query(self, text: str) -> List[float]:
        return [len(text) * 0.1, 0.5, 0.5]

class QdrantVectorStore:
    """
    A production-grade wrapper around QdrantClient for a RAG architecture.
    """
    def __init__(self, collection_name: str, embedding_model: Embeddings, vector_size: int = 3) -> None:
        # Using memory storage for local execution, but in production use path="./qdrant_data" or url="http://localhost:6333"
        self.client = QdrantClient(location=":memory:")
        self.collection_name = collection_name
        self.embedding_model = embedding_model
        
        # Initialize the collection
        self._ensure_collection_exists(vector_size)

    def _ensure_collection_exists(self, vector_size: int) -> None:
        """Creates the Qdrant collection if it does not already exist."""
        if not self.client.collection_exists(collection_name=self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
            )
            print(f"Collection '{self.collection_name}' created.")

    def add_documents(self, documents: List[Document]) -> None:
        """
        Embeds documents and stores them in Qdrant with their metadata and content as payload.
        """
        if not documents:
            return

        texts = [doc.content for doc in documents]
        vectors = self.embedding_model.embed_documents(texts)
        
        points = []
        for doc, vector in zip(documents, vectors):
            payload = {"content": doc.content, **doc.metadata}
            point = PointStruct(id=doc.id, vector=vector, payload=payload)
            points.append(point)
            
        self.client.upsert(
            collection_name=self.collection_name,
            points=points
        )
        print(f"Upserted {len(points)} documents to '{self.collection_name}'.")

    def similarity_search(self, query: str, k: int = 2) -> List[Document]:
        """
        Embeds the query and retrieves the top k most similar documents from Qdrant.
        """
        query_vector = self.embedding_model.embed_query(query)
        
        search_result = self.client.query_points(
            collection_name=self.collection_name,
            query=query_vector,
            limit=k
        )
        
        results = []
        for scored_point in search_result.points:
            payload = scored_point.payload or {}
            content = payload.pop("content", "")
            doc = Document(id=str(scored_point.id), content=content, metadata=payload)
            results.append(doc)
            
        return results

# Implementation Execution
if __name__ == "__main__":
    embedder = MockEmbeddings()
    vector_store = QdrantVectorStore(collection_name="engineering_docs", embedding_model=embedder)
    
    docs_to_add = [
        Document(content="Qdrant is a fast vector database.", metadata={"category": "database"}),
        Document(content="LangChain orchestrates AI agents.", metadata={"category": "framework"}),
        Document(content="Python is the primary language for AI engineering.", metadata={"category": "language"})
    ]
    
    vector_store.add_documents(docs_to_add)
    
    query = "Tell me about vector databases"
    print(f"\nQuerying for: '{query}'")
    results = vector_store.similarity_search(query, k=1)
    
    for idx, res in enumerate(results, 1):
        print(f"Result {idx}: {res.content} (Metadata: {res.metadata})")


Collection 'engineering_docs' created.
Upserted 3 documents to 'engineering_docs'.

Querying for: 'Tell me about vector databases'
Result 1: LangChain orchestrates AI agents. (Metadata: {'category': 'framework'})


## Common Pitfalls in Production

1. **Dimensionality Mismatch:** Trying to insert a vector of dimension 1536 (e.g., from OpenAI `text-embedding-3-small`) into a collection configured for dimension 768. The Qdrant engine will reject the upsert. Always ensure your `VectorParams` size matches your embedding model's output exactly.
2. **Payload Bloat:** Storing massive base64 encoded images or raw multi-megabyte PDFs directly in the Qdrant payload. Vector databases are optimized for search, not as general-purpose object storage. Store document IDs in Qdrant and keep heavy binary files in S3/Blob Storage.
3. **Inconsistent Distance Metrics:** Using `Dot` product distance in your Qdrant configuration but failing to normalize your vectors. While Cosine distance automatically accounts for vector magnitude, Dot product requires explicit normalization. Default to `Cosine` unless you have specific optimization needs.


## Practical Lab / Homework

**Task:** Extend our `QdrantVectorStore` implementation to support metadata filtering during retrieval. 

1. Write a standalone implementation of a `FilteredQdrantVectorStore` that inherits from `QdrantVectorStore`.
2. Add a `filter_dict` parameter to the `similarity_search` method (or create `similarity_search_with_filter`).
3. Translate the `filter_dict` (e.g., `{"category": "database"}`) into Qdrant's `Filter` and `FieldCondition` objects.
4. Execute the code to demonstrate inserting documents and retrieving *only* the documents matching the filter.

**Constraint:** No pseudo-code. Use strict type hinting. Use standard `qdrant_client` models (`Filter`, `FieldCondition`, `MatchValue`).


In [2]:
from typing import Optional
from qdrant_client.http.models import Filter, FieldCondition, MatchValue

class FilteredQdrantVectorStore(QdrantVectorStore):
    """
    Extends the QdrantVectorStore to allow metadata filtering during search.
    """
    def similarity_search_with_filter(self, query: str, filter_dict: Optional[Dict[str, Any]] = None, k: int = 2) -> List[Document]:
        """
        Embeds the query and retrieves documents, strictly filtering by the provided metadata.
        """
        query_vector = self.embedding_model.embed_query(query)
        
        qdrant_filter = None
        if filter_dict:
            # Construct a list of FieldConditions requiring exact matches for the provided dictionary
            conditions = []
            for key, value in filter_dict.items():
                condition = FieldCondition(
                    key=key,
                    match=MatchValue(value=value)
                )
                conditions.append(condition)
                
            qdrant_filter = Filter(must=conditions)
        
        search_result = self.client.query_points(
            collection_name=self.collection_name,
            query=query_vector,
            query_filter=qdrant_filter,
            limit=k
        )
        
        results = []
        for scored_point in search_result.points:
            payload = scored_point.payload or {}
            content = payload.pop("content", "")
            doc = Document(id=str(scored_point.id), content=content, metadata=payload)
            results.append(doc)
            
        return results

# Lab Solution Execution
if __name__ == "__main__":
    embedder = MockEmbeddings()
    filtered_store = FilteredQdrantVectorStore(collection_name="filtered_docs", embedding_model=embedder)
    
    docs_to_add = [
        Document(content="PostgreSQL is a relational database.", metadata={"category": "database", "type": "relational"}),
        Document(content="Qdrant is a vector database.", metadata={"category": "database", "type": "vector"}),
        Document(content="FastAPI is a Python web framework.", metadata={"category": "framework", "type": "web"})
    ]
    
    filtered_store.add_documents(docs_to_add)
    
    # We want to search for databases, but specifically vector databases
    query = "Tell me about databases"
    metadata_filter = {"type": "vector"}
    
    print(f"\nQuerying for: '{query}' with filter {metadata_filter}")
    results = filtered_store.similarity_search_with_filter(query, filter_dict=metadata_filter, k=2)
    
    for idx, res in enumerate(results, 1):
        print(f"Result {idx}: {res.content} (Metadata: {res.metadata})")


Collection 'filtered_docs' created.
Upserted 3 documents to 'filtered_docs'.

Querying for: 'Tell me about databases' with filter {'type': 'vector'}
Result 1: Qdrant is a vector database. (Metadata: {'category': 'database', 'type': 'vector'})
